# MinIO S3 Data Explorer - Gold Layer

In [5]:
# Cài đặt các thư viện cần thiết
import subprocess
import sys

# Tất cả packages cần thiết cho notebook
packages = [
    'minio',
    'pandas',
    'pyarrow',
    'confluent-kafka',
    'fastavro',
    'cachetools',
    'httpx',
    'attrs',
    'authlib',
    'requests'
]

print("[INFO] Cài đặt thư viện...\n")
failed_packages = []

for package in packages:
    try:
        # Thử import package (handle package name khác import name)
        if package == 'confluent-kafka':
            __import__('confluent_kafka')
        elif package == 'pyarrow':
            __import__('pyarrow')
        elif package == 'httpx':
            __import__('httpx')
        elif package == 'cachetools':
            __import__('cachetools')
        else:
            __import__(package)
        print(f"[OK] {package:<25} - đã có sẵn")
    except ImportError:
        print(f"[INSTALL] Đang cài {package:<25}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"[OK] {package:<25} - cài xong")
        except Exception as err:
            print(f"[ERROR] {package:<25} - LỖI: {err}")
            failed_packages.append(package)

if failed_packages:
    print(f"\n[WARN] Các package cài đặt thất bại: {failed_packages}")
else:
    print("\n[OK] Tất cả thư viện sẵn sàng!")

[INFO] Cài đặt thư viện...

[OK] minio                     - đã có sẵn
[OK] pandas                    - đã có sẵn
[OK] pyarrow                   - đã có sẵn
[OK] confluent-kafka           - đã có sẵn
[OK] fastavro                  - đã có sẵn
[OK] cachetools                - đã có sẵn
[OK] httpx                     - đã có sẵn
[OK] attrs                     - đã có sẵn
[OK] authlib                   - đã có sẵn
[OK] requests                  - đã có sẵn

[OK] Tất cả thư viện sẵn sàng!


In [6]:
import os
import io
from minio import Minio
import pandas as pd
import pyarrow.parquet as pq
from datetime import datetime

print("[INFO] Imports hoàn tất!")

[INFO] Imports hoàn tất!


## MinIO Configuration for Gold Layer Exploration

In [8]:
MINIO_CONFIG = {
    'endpoint': 'localhost:9000',
    'access_key': 'minioadmin',
    'secret_key': 'minioadmin',
    'secure': False,
    'region': 'ap-southeast-1'
}

BUCKET_NAME = 'data-lake'
GOLD_PATH_PREFIX = 'warehouse/gold/'

try:
    minio_client = Minio(
        MINIO_CONFIG['endpoint'],
        access_key=MINIO_CONFIG['access_key'],
        secret_key=MINIO_CONFIG['secret_key'],
        secure=MINIO_CONFIG['secure'],
        region=MINIO_CONFIG['region']
    )
    
    # Kiểm tra kết nối
    if minio_client.bucket_exists(BUCKET_NAME):
        print(f"[SUCCESS] Kết nối MinIO thành công! Bucket '{BUCKET_NAME}' đã sẵn sàng.")
    else:
        print(f"[ERROR] Bucket '{BUCKET_NAME}' không tồn tại.")
    
except Exception as e:
    print(f"[ERROR] Kết nối MinIO thất bại: {e}")

[SUCCESS] Kết nối MinIO thành công! Bucket 'data-lake' đã sẵn sàng.


## 📊 Overview of the tables in the Gold layer

There are two main mart tables in the Gold layer:
1. **mart_sales_overview**: sales details table (denormalized).
2. **mart_customer_lifetime_value**: customer behavior table.

In [20]:
def list_gold_tables():
    try:
        objects = minio_client.list_objects(BUCKET_NAME, prefix=GOLD_PATH_PREFIX, recursive=True)
        tables = {}
        for obj in objects:
            parts = obj.object_name.split('/')
            if len(parts) >= 3:
                table_name = parts[2]
                if table_name not in tables:
                    tables[table_name] = {'count': 0, 'size_mb': 0}
                tables[table_name]['count'] += 1
                tables[table_name]['size_mb'] += obj.size / (1024 * 1024)
        return tables
    except Exception as e:
        print(f"Lỗi: {e}")
        return {}

gold_tables = list_gold_tables()
print("\nBảng Gold hiện có trong MinIO:")
print("=" * 50)
for name, info in gold_tables.items():
    print(f"- {name:<30} | {info['count']:>3} objects | {info['size_mb']:>6.2f} MB")


Bảng Gold hiện có trong MinIO:
- mart_customer_lifetime_value   |   6 objects |   0.02 MB
- mart_sales_overview            |   6 objects |   0.28 MB


## Explore the detailed data.


In [30]:
def read_table_sample(table_name, limit=50):
    prefix = f"{GOLD_PATH_PREFIX}{table_name}/data/"
    try:
        objects = list(minio_client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True))
        parquet_files = [obj.object_name for obj in objects if obj.object_name.endswith('.parquet')]
        
        if not parquet_files:
            print(f"[WARN] Không tìm thấy file dữ liệu cho bảng {table_name}")
            return
        
        # Đọc file đầu tiên
        response = minio_client.get_object(BUCKET_NAME, parquet_files[0])
        df = pd.read_parquet(io.BytesIO(response.read()))
        
        print(f"\n{'='*80}")
        print(f"📊 TABLE: {table_name.upper()}")
        print(f"{'='*80}")
        print(f"\n[SCHEMA]:")
        print(df.dtypes)
        print(f"\n[SAMPLE - Top {limit} rows]:")
        display(df.head(limit))
        
    except Exception as e:
        print(f"[ERROR] Lỗi khi đọc bảng {table_name}: {e}")

for table in gold_tables.keys():
    read_table_sample(table)


📊 TABLE: MART_CUSTOMER_LIFETIME_VALUE

[SCHEMA]:
customer_id         object
total_orders         int64
total_spent_usd    float64
first_order         object
last_order          object
dtype: object

[SAMPLE - Top 50 rows]:


,customer_id,total_orders,total_spent_usd,first_order,last_order
0,usr_2vloje4g,35,94002.0619,2026-04-13,2026-04-13
1,usr_c1oj4q4j,28,97401.3055,2026-04-13,2026-04-13
2,usr_0xjdbznd,36,95851.9455,2026-04-13,2026-04-13
3,usr_r4u619xv,34,80074.0164,2026-04-13,2026-04-13
4,usr_dc2aw221,32,117290.0654,2026-04-13,2026-04-13
5,usr_rakxwnee,42,121453.8909,2026-04-13,2026-04-13
6,usr_p2tgcdlh,38,125313.0273,2026-04-13,2026-04-13
7,usr_gum3xwuv,36,113007.0135,2026-04-13,2026-04-13
8,usr_gnhfld6w,26,81670.8959,2026-04-13,2026-04-13
9,usr_xutmuiyg,32,90253.8964,2026-04-13,2026-04-13



📊 TABLE: MART_SALES_OVERVIEW

[SCHEMA]:
order_id           object
order_date         object
customer_id        object
country            object
product_name       object
category           object
quantity            int32
amount_usd        float64
payment_status     object
dtype: object

[SAMPLE - Top 50 rows]:


,order_id,order_date,customer_id,country,product_name,category,quantity,amount_usd,payment_status
0,ord_002iybk3cq,2026-04-13,usr_jajamyka,SG,Product 124,Electronics,6,1875.1800,Unknown
1,ord_005atj1uru,2026-04-13,usr_zwek9oav,MM,Product 72,Beauty,13,6208.3840,Unknown
2,ord_006o6s3rnx,2026-04-13,usr_zojfgd1l,SG,Product 19,Fashion,2,73.5744,SETTLED
3,ord_00glvr7dyl,2026-04-13,usr_hjveuepy,PH,Product 178,Beauty,5,1354.7000,SETTLED
4,ord_00lfn07p0u,2026-04-13,usr_tr93kngj,VN,Product 41,Home,19,5180.3367,SETTLED
5,ord_00qfhv1vea,2026-04-13,usr_s19kheho,LA,Product 35,Fashion,2,937.5060,PENDING
6,ord_012i7670u0,2026-04-13,usr_w2ardi2r,TH,Product 15,Electronics,9,960.3387,FAILED
7,ord_0151b98xg0,2026-04-13,usr_fgzz1b7g,ID,Product 69,Beauty,3,1593.9456,SETTLED
8,ord_01beupyqkr,2026-04-13,usr_k470msww,MM,Product 50,Beauty,2,1046.1528,Unknown
9,ord_01e7qesxzz,2026-04-13,usr_3ab9lq5c,BN,Product 28,Beauty,14,2086.2450,PENDING
